# Stage 02 — Data Preparation

Load raw data, merge datasets, construct variables, write `data/dataset.parquet`.
Follow `skills/stage_02.md` for detailed instructions.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd

spec = json.loads(PAPER_SPEC.read_text())
print('Identification type:', spec['identification']['type'])
print('Primary file:', spec['identification']['primary_data_file'])
print('Secondary datasets:', spec['identification'].get('secondary_datasets', []))
print('Data filter:', spec['identification'].get('data_filter', 'None'))

Identification type: OLS
Primary file: data_tariffs.csv
Secondary datasets: []
Data filter: industry == 211212


In [2]:
# --- Load primary dataset (CSV) ---
primary_file = RAW_DATA_DIR / spec['identification']['primary_data_file']
df_raw = pd.read_csv(str(primary_file))
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {list(df_raw.columns)}')
df_raw.head()

Raw dataset shape: (1134, 53)
Columns (53): ['Unnamed: 0', 'wbcode', 'industry', 'industry_description', 'init_year', 'final_year', 'skill_2_1', 'skill1_corr', 'diffa', 'diffg', 'tskilla', 'tnoskilla', 'tskillg', 'tnoskillg', 'avg_tar', 'init_tar', 'prod_growth', 'growth', 'inv', 'init', 'human_cap', 'dum80_83', 'dum85_87', 'lat_america', 'e_europe', 'w_africa', 'e_africa', 's_c_africa', 'n_afr_me', 'e_asia', 's_e_asia', 's_w_asia', 'ln_init_q', 'ln_init_q_skilla', 'ln_init_q_unskilla', 'ln_init_q_skillg', 'ln_init_q_unskillg', 'voice2000', 'political_stability2000', 'govt_effectiveness2000', 'reg_quality2000', 'rule_of_law2000', 'control_corrup2000', 'growth_human_cap', 'patent_growth', 'diff_growth_q_a', 'diff_growth_q_g', 'fin_diffa', 'fin_diffg', 'percent_firms_connected', 'ln_viol_per_car', 'gatt_year', 'wto_year']


,Unnamed: 0,wbcode,industry,industry_description,init_year,final_year,skill_2_1,skill1_corr,diffa,diffg,...,growth_human_cap,patent_growth,diff_growth_q_a,diff_growth_q_g,fin_diffa,fin_diffg,percent_firms_connected,ln_viol_per_car,gatt_year,wto_year
0,1,ARG,249,Professional equip,1987,2000,0.797,-0.450096,-0.271852,-0.40546,...,0.05207,0.179712,0.036627,0.047094,0.248441,0.752458,0.0,1.80234,1967.0,1995.0
1,2,ARG,245,Textiles/clothing,1987,2000,0.132,-0.450096,-0.271852,-0.40546,...,0.05207,0.179712,0.036627,0.047094,0.248441,0.752458,0.0,1.80234,1967.0,1995.0
2,3,ARG,231,Non-elec machinery,1987,2000,0.414,-0.450096,-0.271852,-0.40546,...,0.05207,0.179712,0.036627,0.047094,0.248441,0.752458,0.0,1.80234,1967.0,1995.0
3,4,ARG,244,Paper prod,1987,2000,0.397,-0.450096,-0.271852,-0.40546,...,0.05207,0.179712,0.036627,0.047094,0.248441,0.752458,0.0,1.80234,1967.0,1995.0
4,5,ARG,140,Mineral fuels,1987,2000,0.593,-0.450096,-0.271852,-0.40546,...,0.05207,0.179712,0.036627,0.047094,0.248441,0.752458,0.0,1.80234,1967.0,1995.0


In [3]:
# --- Apply data filter: industry == 211212 (country-level observations) ---
data_filter = spec['identification'].get('data_filter', None)
print(f'Data filter: {data_filter}')

n_before = len(df_raw)
df = df_raw[df_raw['industry'] == 211212].copy()
n_after = len(df)
print(f'Rows before filter: {n_before}')
print(f'Rows after filter (industry == 211212): {n_after}')
print(f'Rows removed: {n_before - n_after}')
assert n_after == 63, f'Expected 63 observations after filter, got {n_after}'
print(f'\n✓ Filter applied: N = {n_after} (matches expected 63)')

Data filter: industry == 211212
Rows before filter: 1134
Rows after filter (industry == 211212): 63
Rows removed: 1071

✓ Filter applied: N = 63 (matches expected 63)


In [4]:
# --- Verify all required variables exist ---
outcome = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
additional_treatments = [t['variable'] for t in spec['identification'].get('additional_treatments', [])]
controls = spec['identification']['controls']

all_treatments = [treatment] + additional_treatments
all_required = [outcome] + all_treatments + controls

print(f'Outcome: {outcome}')
print(f'Primary treatment: {treatment}')
print(f'Additional treatments: {additional_treatments}')
print(f'Controls ({len(controls)}): {controls}')
print(f'Total required variables: {len(all_required)}')

missing = [v for v in all_required if v not in df.columns]
assert len(missing) == 0, f'Missing variables: {missing}'
print(f'\n✓ All {len(all_required)} required variables present (1 outcome + 3 treatments + 17 controls)')

# No variable construction needed:
# - functional_form is "linear" (no squared terms)
# - no fixed_effects requiring dummies (region dummies already in data)
# - no instrument
# - no log transformation needed (outcome "growth" is already log annual growth)

Outcome: growth
Primary treatment: skill1_corr
Additional treatments: ['diffa', 'diffg']
Controls (17): ['avg_tar', 'init', 'inv', 'human_cap', 'w_africa', 'e_africa', 's_c_africa', 'n_afr_me', 'e_europe', 'lat_america', 'e_asia', 's_e_asia', 's_w_asia', 'dum80_83', 'dum85_87', 'ln_init_q_skilla', 'ln_init_q_unskilla']
Total required variables: 21

✓ All 21 required variables present (1 outcome + 3 treatments + 17 controls)


In [5]:
# --- Validate: missingness and descriptive statistics ---
print('=== Missingness in key variables ===')
print(df[all_required].isnull().sum())
print(f'\nTotal rows with all key vars present: {df[all_required].dropna().shape[0]}')

n_complete = df[all_required].dropna().shape[0]
assert n_complete >= 30, f'Too few complete observations: {n_complete}'
assert n_complete == 63, f'Expected 63 complete cases, got {n_complete}'
print(f'✓ All 63 observations have complete data (no listwise deletion needed)')

print('\n=== Descriptive Statistics ===')
df[all_required].describe().round(4)

=== Missingness in key variables ===
growth                0
skill1_corr           0
diffa                 0
diffg                 0
avg_tar               0
init                  0
inv                   0
human_cap             0
w_africa              0
e_africa              0
s_c_africa            0
n_afr_me              0
e_europe              0
lat_america           0
e_asia                0
s_e_asia              0
s_w_asia              0
dum80_83              0
dum85_87              0
ln_init_q_skilla      0
ln_init_q_unskilla    0
dtype: int64

Total rows with all key vars present: 63
✓ All 63 observations have complete data (no listwise deletion needed)

=== Descriptive Statistics ===


,growth,skill1_corr,diffa,diffg,avg_tar,init,inv,human_cap,w_africa,e_africa,...,n_afr_me,e_europe,lat_america,e_asia,s_e_asia,s_w_asia,dum80_83,dum85_87,ln_init_q_skilla,ln_init_q_unskilla
count,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,...,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000,63.0000
mean,0.0180,-0.2850,-0.4239,-0.6428,-1.8424,8.3843,2.7877,-2.3378,0.0794,0.0794,...,0.0952,0.0317,0.1587,0.0635,0.0635,0.0794,0.4762,0.2063,22.3677,23.5638
std,0.0194,0.2516,0.5514,0.5936,1.4377,1.0341,0.6458,1.2325,0.2725,0.2725,...,0.2959,0.1767,0.3684,0.2458,0.2458,0.2725,0.5034,0.4079,2.4864,1.5239
min,-0.0255,-0.7215,-1.4092,-2.1863,-9.2103,6.3994,0.6134,-5.0299,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,16.3076,20.4974
25%,0.0079,-0.4711,-0.8123,-1.0657,-2.1389,7.6001,2.5338,-2.8254,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,20.6725,22.3771
50%,0.0198,-0.3385,-0.4644,-0.6055,-1.5147,8.5812,2.8805,-2.1849,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,22.5037,23.5833
75%,0.0275,-0.0590,-0.0632,-0.1291,-1.0378,9.4150,3.2600,-1.4965,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,24.2171,24.5427
max,0.0628,0.1877,1.2358,0.4076,0.5443,9.7805,3.9497,0.1446,1.0000,1.0000,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,27.7575,27.2368


In [6]:
# --- Write output: save ALL original columns (filtered) to parquet ---
df.to_parquet(str(DATASET_PATH), index=False)
print(f'✓ Written: {DATASET_PATH}')
print(f'  Shape: {df.shape}')
print(f'  N observations: {len(df)}')
print(f'  N columns: {len(df.columns)}')
print(f'  Columns: {list(df.columns)}')

✓ Written: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\dataset.parquet
  Shape: (63, 53)
  N observations: 63
  N columns: 53
  Columns: ['Unnamed: 0', 'wbcode', 'industry', 'industry_description', 'init_year', 'final_year', 'skill_2_1', 'skill1_corr', 'diffa', 'diffg', 'tskilla', 'tnoskilla', 'tskillg', 'tnoskillg', 'avg_tar', 'init_tar', 'prod_growth', 'growth', 'inv', 'init', 'human_cap', 'dum80_83', 'dum85_87', 'lat_america', 'e_europe', 'w_africa', 'e_africa', 's_c_africa', 'n_afr_me', 'e_asia', 's_e_asia', 's_w_asia', 'ln_init_q', 'ln_init_q_skilla', 'ln_init_q_unskilla', 'ln_init_q_skillg', 'ln_init_q_unskillg', 'voice2000', 'political_stability2000', 'govt_effectiveness2000', 'reg_quality2000', 'rule_of_law2000', 'control_corrup2000', 'growth_human_cap', 'patent_growth', 'diff_growth_q_a', 'diff_growth_q_g', 'fin_diffa', 'fin_diffg', 'percent_firms_connected', 'ln_viol_per_car', 'gatt_year', 'wto_year']
